# Expanding with Large Language Models: Personalization, Temperature & Responsible AI

**Expanding** is the task of taking shorter input text—such as bullet points, rough instructions, or structured attributes—and having a Large Language Model generate longer, coherent content such as personalized emails, essays, or contextual customer service responses.

### Core Concepts Covered:
1. **Conditioned Expansion:** Generating context-rich, empathetic emails tailored to both the raw customer feedback and inferred sentiment.
2. **Grounding & Detail Anchoring:** Instructing the LLM to extract and address concrete customer pain points rather than generating generic corporate responses.
3. **Decoding Mechanics & the `temperature` Parameter:** Understanding next-token probability distribution, greedy vs. stochastic sampling, and deterministic vs. creative outputs.
4. **Responsible AI & Disclosure:** Enforcing AI transparency by signing bot communications explicitly as an AI agent and avoiding spam/harmful automation.

## 1. Setup & Multi-Provider Client Configuration

Load environment variables from your local `.env` file.
- Supports standard OpenAI credentials (`OPENAI_API_KEY`).
- Fully compatible with Google Gemini via its OpenAI-compatible endpoint (`GEMINI_API_KEY` directed to `https://generativelanguage.googleapis.com/v1beta/openai/`).

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# Load secret API keys from the local .env file
_ = load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GEMINI_API_KEY")
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/" if os.getenv("GEMINI_API_KEY") and not os.getenv("OPENAI_API_KEY") else None

if base_url:
    client = OpenAI(api_key=api_key, base_url=base_url)
else:
    client = OpenAI(api_key=api_key)

### Helper Function: `get_completion`
Encapsulates chat completion requests. We expose `temperature` as a direct argument so we can toggle between deterministic evaluation and creative variation.

In [ ]:
# Active models alias or fixed version fallback
DEFAULT_MODEL = "gemini-flash-latest" if base_url else "gpt-3.5-turbo"

def get_completion(prompt, model=DEFAULT_MODEL, temperature=0):
    """
    Helper function to send single-turn user prompts to the chat completions API.
    - temperature=0: Deterministic and reproducible (greedy decoding).
    - temperature>0: Stochastic and diverse (probabilistic sampling).
    """
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

## 2. Customer Feedback Data & Inferred Context

Here we take a lengthy, multi-faceted customer review for a blender that discusses seasonal price surges, mechanical motor failure after warranty expiration, and food preparation tips.

Following the **inferring** workflow from earlier, the overall customer sentiment is tagged as `negative`.

In [ ]:
# Inferred sentiment from our classification pipeline
sentiment = "negative"

# Customer review text for a kitchen blender
review = """
So, they still had the 17 piece system on seasonal 
sale for around $49 in the month of November, about 
half off, but for some reason (call it price gouging) 
around the second week of December the prices all went 
up to about anywhere from between $70-$89 for the same 
system. And the 11 piece system went up around $10 or 
so in price also from the earlier sale price of $29. 
So it looks okay, but if you look at the base, the part 
where the blade locks into place doesn’t look as good 
as in previous editions from a few years ago, but I 
plan to be very gentle with it (example, I crush 
very hard items like beans, ice, rice, etc. in the 
blender first then pulverize them in the serving size 
I want in the blender then switch to the whipping 
blade for a finer flour, and use the cross cutting blade 
first when making smoothies, then use the flat blade 
if I need them finer/less pulpy). Special tip when making 
smoothies, finely cut and freeze the fruits and 
vegetables (if using spinach-lightly stew soften the 
spinach then freeze until ready for use-and if making 
sorbet, use a small to medium sized food processor) 
that you plan to use that way you can avoid adding so 
much ice if at all-when making your smoothie. 
After about a year, the motor was making a funny noise. 
I called customer service but the warranty expired 
already, so I had to buy another one. FYI: The overall 
quality has gone done in these types of products, so 
they are kind of counting on brand recognition and 
consumer loyalty to maintain sales. Got it in about 
two days.
"""

## 3. Conditioned Customer Support Expansion

### Prompt Strategy: Conditional Branching & AI Transparency
To produce a high-utility customer service response:
1. **Conditional Action:** If sentiment is positive/neutral, thank the user; if negative, apologize and direct them to live support.
2. **Entity Grounding:** Explicitly require the model to cite specific details from the review (e.g., motor failure, pricing fluctuations).
3. **Responsible AI Disclosure:** Mandate that the model sign off transparently as `AI customer agent` so human customers are never deceived.

In [ ]:
prompt_email = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by triple backticks, 
generate a reply to thank the customer for their review.

If the sentiment is positive or neutral, thank them for their review.
If the sentiment is negative, apologize and suggest that they can 
reach out to customer service. 

Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.

Customer review: ```{review}```
Review sentiment: {sentiment}
"""

response_deterministic = get_completion(prompt_email, temperature=0)
print("--- Deterministic Expansion (temperature=0) ---")
print(response_deterministic)

## 4. Understanding & Tuning the `temperature` Parameter

LLMs do not generate full sentences at once; they iteratively compute a probability distribution over the vocabulary to select the next token:

$$\text{Prompt: "My favorite food is "} \longrightarrow \begin{cases} \text{"pizza"} & (53\%) \\ \text{"sushi"} & (30\%) \\ \text{"tacos"} & (5\%) \end{cases}$$

- **`temperature = 0` (Greedy Search):** The model always selects the argmax token (highest probability). Outputs are deterministic, repeatable, and optimal for production classification, mathematical tasks, or data transformations.
- **`temperature > 0` (Probabilistic Sampling):** Flattens the probability distribution. Lower-probability tokens have a chance to be sampled, yielding greater lexical variety, creative phrasing, and diverse suggestions.

### Comparing Successive Runs at `temperature = 0.7`

In [ ]:
print("--- Stochastic Run 1 (temperature=0.7) ---")
response_run_1 = get_completion(prompt_email, temperature=0.7)
print(response_run_1)

print("\n" + "="*60 + "\n")

print("--- Stochastic Run 2 (temperature=0.7) ---")
response_run_2 = get_completion(prompt_email, temperature=0.7)
print(response_run_2)

## 5. Temperature Sweep Experimentation

To observe how temperature controls the degree of exploration, we test the exact same generation task across a range of values ($0.0$, $0.3$, $0.7$, and $1.0$).

In [ ]:
test_prompt = "Write a one-sentence creative opening line for a science fiction novel."
temp_values = [0.0, 0.3, 0.7, 1.0]

for t in temp_values:
    print(f"--- Output at Temperature = {t} ---")
    result = get_completion(test_prompt, temperature=t)
    print(result.strip())
    print("-" * 40)

## 6. Responsible AI & Application Safety Guidelines

While expanding allows software to produce rich prose quickly, generative capabilities present clear societal and operational risks:

1. **Spam & Mass Synthesis Prevention:** Never configure automated loops to flood social forums, marketing channels, or email inboxes with synthetic text.
2. **Clear Provenance & Transparency:** In consumer-facing products, disclose synthetic origin explicitly (e.g., signing off as `AI customer agent` or labeling AI-assisted drafts).
3. **Human-in-the-Loop Review:** For high-stakes communication (medical, financial, legal, or severe customer disputes), use expansion to produce **drafts** for human approval rather than sending directly into autonomous execution.

## 7. Strategic Summary & Temperature Cheat Sheet

| Temperature Setting | Sampling Behavior | Best Use Cases |
| :--- | :--- | :--- |
| **`0.0`** | Deterministic / Greedy (Top token only) | Data extraction, classification, math, code generation, reproducible tests. |
| **`0.1 - 0.3`** | Low randomness (Focused & conservative) | Business communications, technical documentation, consistent summarization. |
| **`0.7 - 0.8`** | Moderate randomness (Creative yet coherent) | Brainstorming, drafting marketing copy, conversational chatbot variety. |
| **`1.0+`** | High randomness (Wide token distribution) | Fiction writing, poetry, divergent lateral thinking (higher risk of incoherence). |